### Import CSV Dataset

In [ ]:
import pandas as pd  # pandas for data manipulation and DataFrame operations
import re            # re (regex) for pattern matching to validate IP format

# display is better than print for wider tables
from IPython.display import display

# Load the dataset from the current directory into a pandas DataFrame
data = pd.read_csv("./demo_dataset.csv")

### Check for Invalid IP Addresses

In [ ]:
def is_valid_ip(ip):
    """
    Validates whether a given value is a properly formatted IPv4 address.
    Returns False for NaN/null values, True for valid IPs, False for invalid ones.
    """

    # First, handle missing/null values (NaN) — regex cannot process floats,
    # so we catch them early and treat them as invalid
    if pd.isna(ip):
        return False

    # Define the IPv4 validation regex pattern:
    # Each octet must be:
    #   25[0-5]        → 250-255
    #   2[0-4][0-9]    → 200-249
    #   [01]?[0-9][0-9]? → 0-199
    # Four octets separated by dots, anchored with ^ (start) and $ (end)
    pattern = re.compile(
        r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}'  # first 3 octets with dot
        r'(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'          # last octet, no dot
    )

    # Convert to string (safety net for any non-string types) and match against pattern
    # bool() converts the match object to True (match found) or False (no match)
    return bool(pattern.match(str(ip)))

# Apply is_valid_ip to every value in the 'source_ip' column
# ~ (tilde) is the NOT operator — selects rows where is_valid_ip returned False
# This gives us all rows with missing, malformed, or out-of-range IP addresses
invalid_ips = data[~data['source_ip'].apply(is_valid_ip)]

# Print the full rows of invalid entries for inspection
display(invalid_ips)

### Check for Invalid Ports

In [ ]:
def is_valid_port(port):
    """
    Validates whether a given value is a valid network port number.
    Valid ports are integers in the range 0–65535 (defined by TCP/UDP spec).
    Returns True for valid ports, False for anything else.
    """

    try:
        # Attempt to cast the value to an integer
        # This handles cases where the port comes in as a float (e.g. 80.0)
        # or a numeric string (e.g. "443") — both will convert cleanly
        port = int(port)

        # Check if the port falls within the valid range:
        # 0     → reserved but technically valid
        # 1–1023    → well-known ports (HTTP, FTP, SSH, etc.)
        # 1024–49151  → registered ports
        # 49152–65535 → dynamic/private ports
        return 0 <= port <= 65535

    except (ValueError, TypeError):
        # ValueError → non-numeric strings like "STRING_PORT", "UNUSED_PORT"
        # TypeError  → NaN/null values which can't be converted to int
        return False

# Apply is_valid_port to every value in the 'destination_port' column
# ~ (tilde) is the NOT operator — selects rows where is_valid_port returned False
# This catches string placeholders, nulls, and out-of-range numbers
invalid_ports = data[~data['destination_port'].apply(is_valid_port)]

# Print the full rows of invalid port entries for inspection
display(invalid_ports)

### Check for Invalid Protocols

In [ ]:
# Define the set of accepted network protocols
# Using a set instead of a list for O(1) lookup performance
VALID_PROTOCOLS = {'TCP', 'TLS', 'SSH', 'POP3', 'DNS', 'HTTPS', 'SMTP', 'FTP', 'UDP', 'HTTP'}

def is_valid_protocol(protocol):
    """
    Validates whether a given value is a recognized/allowed network protocol.
    Returns True if the protocol is in the approved list, False otherwise.
    """

    # Handle missing/null values — NaN cannot be compared to strings
    if pd.isna(protocol):
        return False

    # Convert to uppercase string to make the check case-insensitive
    # e.g. 'http', 'Http', 'HTTP' all pass
    protocol = str(protocol).strip().upper()

    # Check if the protocol exists in our approved set
    # Set lookup is O(1) — faster than list .isin() for large datasets
    return protocol in VALID_PROTOCOLS

# Apply is_valid_protocol to every value in the 'protocol' column
# ~ (tilde) is the NOT operator — selects rows where is_valid_protocol returned False
# This catches null values, typos, or completely invalid protocol names
invalid_protocols = data[~data['protocol'].apply(is_valid_protocol)]

# Print the full rows of invalid protocol entries for inspection
display(invalid_protocols)

### Check for Invalid Bytes Transferred

In [ ]:
def is_valid_bytes(value):
    """
    Validates whether a given value is a valid bytes_transferred amount.
    Valid values are non-negative integers (0 or above).
    Returns True for valid, False for negative, non-numeric, or null values.
    """

    try:
        # Attempt to cast to int — handles numeric strings like "1024"
        # Using 'value' instead of 'bytes' since 'bytes' is a Python built-in
        value = int(value)

        # Bytes transferred must be 0 or positive — negative makes no sense
        return value >= 0

    except (ValueError, TypeError):
        # ValueError → non-numeric strings like "NON_NUMERIC", "NEGATIVE"
        # TypeError  → NaN/null values which can't be converted to int
        return False

# Apply is_valid_bytes to every value in the 'bytes_transferred' column
# ~ (tilde) is the NOT operator — selects rows where is_valid_bytes returned False
invalid_bytes = data[~data['bytes_transferred'].apply(is_valid_bytes)]

# Display as formatted HTML table
display(invalid_bytes)

### Checking for Invalid Threat Levels

In [ ]:
def is_valid_threat_level(threat_level):
    """
    Validates whether a given value is a valid threat level.
    Valid threat levels are integers 0, 1, or 2:
        0 → No threat
        1 → Low/Medium threat
        2 → High threat
    Returns True for valid, False for out-of-range, non-numeric, or null values.
    """

    try:
        # Attempt to cast to int — handles numeric strings like "0", "1", "2"
        threat_level = int(threat_level)

        # Threat level must be within the defined range of 0 to 2
        return 0 <= threat_level <= 2

    except (ValueError, TypeError):
        # ValueError → non-numeric strings like "?" or "UNKNOWN"
        # TypeError  → NaN/null values which can't be converted to int
        return False

# Apply is_valid_threat_level to every value in the 'threat_level' column
# ~ (tilde) is the NOT operator — selects rows where is_valid_threat_level returned False
invalid_threat_levels = data[~data['threat_level'].apply(is_valid_threat_level)]

# Display as formatted HTML table
display(invalid_threat_levels)

### Drop Invalid Entries

In [ ]:
# ── Drop Invalid Rows ────────────────────────────────────────────────────────
# Remove all rows flagged by our validation functions.
# errors='ignore' prevents crashes when the same row appears in multiple
# invalid sets (e.g. a row with both a bad IP AND a bad port) — pandas
# simply skips already-dropped indexes instead of throwing a KeyError.
# After dropping the bad data from our dataset, we are only left with 77 clean entries.

data = data.drop(invalid_ips.index,           errors='ignore')  # drop bad IPs
data = data.drop(invalid_ports.index,         errors='ignore')  # drop bad ports
data = data.drop(invalid_protocols.index,     errors='ignore')  # drop bad protocols
data = data.drop(invalid_bytes.index,         errors='ignore')  # drop bad byte values
data = data.drop(invalid_threat_levels.index, errors='ignore')  # drop bad threat levels

# ── Verify Clean Dataset ─────────────────────────────────────────────────────
# describe(include='all') gives a statistical summary of every column:
# - For numeric columns: count, mean, std, min, max, quartiles
# - For string columns:  count, unique, top (most frequent), freq
# Use this to confirm no invalid values remain and check data distribution
display(data.describe(include='all'))

print(f"Rows removed: {100 - len(data)}")
print(f"Clean rows remaining: {len(data)}")

### Imputing Missing Values

In [ ]:
import pandas as pd    # data manipulation and DataFrame operations
import numpy as np     # numerical operations and NaN representation
import re              # regex for IP address pattern validation
from ipaddress import ip_address          # standard library IP validation (unused here but useful for extension)
from sklearn.impute import SimpleImputer  # ML-grade imputation for numeric and categorical columns

# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('demo_dataset.csv')

# ── Define Invalid Placeholder Values ────────────────────────────────────────
# These are the known corrupt/placeholder strings we identified during validation
# grouping them by column type makes the code readable and easy to extend
invalid_ips    = ['INVALID_IP', 'MISSING_IP']           # non-routable string placeholders in source_ip
invalid_ports  = ['STRING_PORT', 'UNUSED_PORT']         # non-numeric placeholders in destination_port
invalid_bytes  = ['NON_NUMERIC', 'NEGATIVE']            # invalid byte transfer values
invalid_threat = ['?']                                  # unknown/unclassified threat levels

# ── Standardize Invalid Values to NaN ────────────────────────────────────────
# Replace all known invalid placeholders with NaN in a single pass.
# This gives us a uniform representation of "missing" across all columns,
# which is required for imputers and most ML pipelines to work correctly.
df.replace(invalid_ips + invalid_ports + invalid_bytes + invalid_threat, np.nan, inplace=True)

# ── Coerce Numeric Columns ────────────────────────────────────────────────────
# After replacing string placeholders, some columns are still stored as 'object' dtype.
# pd.to_numeric with errors='coerce' converts valid numbers and turns any remaining
# non-numeric stragglers into NaN — a safe catch-all for anything we may have missed.
df['destination_port']  = pd.to_numeric(df['destination_port'],  errors='coerce')
df['bytes_transferred'] = pd.to_numeric(df['bytes_transferred'], errors='coerce')
df['threat_level']      = pd.to_numeric(df['threat_level'],      errors='coerce')

# ── Validate & Nullify Invalid IP Addresses ───────────────────────────────────
def is_valid_ip(ip):
    """
    Validates IPv4 format using regex.
    Returns the original IP string if valid, or NaN if missing/malformed.
    This converts bad IPs (e.g. 192.168.1.1100, 10.0.0.300) into NaN
    so they can be handled uniformly in the imputation step.
    """
    pattern = re.compile(
        r'^((25[0-5]|2[0-4][0-9]|[01]?\d?\d)\.){3}'  # first 3 octets
        r'(25[0-5]|2[0-4]\d|[01]?\d?\d)$'             # last octet
    )
    # Return NaN if the value is already NaN or fails the regex check
    if pd.isna(ip) or not pattern.match(str(ip)):
        return np.nan
    return ip  # return original valid IP string unchanged

# Apply validation — invalid IPs become NaN, ready for imputation
df['source_ip'] = df['source_ip'].apply(is_valid_ip)

# ── Impute Missing Values ─────────────────────────────────────────────────────
# Define which columns are numeric and which are categorical
# as each type requires a different imputation strategy
numeric_cols     = ['destination_port', 'bytes_transferred', 'threat_level']
categorical_cols = ['protocol']

# Numeric imputation — use MEDIAN (more robust than mean for skewed data
# or datasets with outliers like 999999 we saw in destination_port)
num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Categorical imputation — use MOST FREQUENT (mode)
# fills missing protocol values with the most common protocol in the dataset
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# ── Verify Result ─────────────────────────────────────────────────────────────
# Confirm no NaN values remain in the imputed columns
display(df.isnull().sum())
display(df.describe(include='all'))

### Sophisticated Imputation

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.impute import KNNImputer

# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('demo_dataset.csv')

# ── Define Invalid Placeholder Values ────────────────────────────────────────
invalid_ips    = ['INVALID_IP', 'MISSING_IP']
invalid_ports  = ['STRING_PORT', 'UNUSED_PORT']
invalid_bytes  = ['NON_NUMERIC', 'NEGATIVE']
invalid_threat = ['?']

# ── Standardize Invalid Values to NaN ────────────────────────────────────────
df.replace(invalid_ips + invalid_ports + invalid_bytes + invalid_threat, np.nan, inplace=True)

# ── Coerce Numeric Columns ────────────────────────────────────────────────────
df['destination_port']  = pd.to_numeric(df['destination_port'],  errors='coerce')
df['bytes_transferred'] = pd.to_numeric(df['bytes_transferred'], errors='coerce')
df['threat_level']      = pd.to_numeric(df['threat_level'],      errors='coerce')

# ── Validate & Nullify Invalid IP Addresses ───────────────────────────────────
def is_valid_ip(ip):
    """
    Validates IPv4 format using regex.
    Returns the original IP string if valid, NaN if missing or malformed.
    """
    pattern = re.compile(
        r'^((25[0-5]|2[0-4][0-9]|[01]?\d?\d)\.){3}'
        r'(25[0-5]|2[0-4]\d|[01]?\d?\d)$'
    )
    if pd.isna(ip) or not pattern.match(str(ip)):
        return np.nan
    return ip

df['source_ip'] = df['source_ip'].apply(is_valid_ip)

# ── Advanced Imputation with KNNImputer ───────────────────────────────────────
# Unlike SimpleImputer (mean/median/mode), KNNImputer considers relationships
# between features to estimate missing values from the K most similar rows.
# e.g. a row with protocol=FTP is more likely to have port=21 than port=80 —
# KNN picks up on those patterns from the existing data.
#
# n_neighbors=5 → uses the 5 most similar rows to estimate each missing value
# Higher K = smoother results | Lower K = more locally specific
#
# ── Optional: IterativeImputer (even more advanced) ──────────────────────────
# Models each feature as a function of all others, iterating until convergence.
# Similar to MICE (Multiple Imputation by Chained Equations).
# Useful when features are highly correlated with each other.
#
# from sklearn.experimental import enable_iterative_imputer  # must enable first
# from sklearn.impute import IterativeImputer
# iter_imputer = IterativeImputer(max_iter=10, random_state=42)
# df[numeric_cols] = iter_imputer.fit_transform(df[numeric_cols])

# KNN only works on numeric columns — source_ip and protocol are excluded
numeric_cols     = ['destination_port', 'bytes_transferred', 'threat_level']
categorical_cols = ['protocol']

# Impute numeric columns with KNN
knn_imputer = KNNImputer(n_neighbors=5)
df[numeric_cols] = knn_imputer.fit_transform(df[numeric_cols])

# Protocol still uses most_frequent — KNN can't handle categorical data
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# ── Verify Result ─────────────────────────────────────────────────────────────
display(df.isnull().sum())
display(df.describe(include='all'))

### Post-imputation Cleanup/Correction

In [ ]:
# ── Post-Imputation Cleanup & Correction ──────────────────────────────────────
# After imputation, apply final business-rule corrections to ensure
# all values are not just non-null, but also semantically valid.

# ── 1. Protocol Correction ────────────────────────────────────────────────────
# KNN/SimpleImputer may have introduced or left protocols outside our known list.
# Replace any invalid protocol with the most frequent (mode) value in the column.
valid_protocols = ['TCP', 'TLS', 'SSH', 'POP3', 'DNS', 'HTTPS', 'SMTP', 'FTP', 'UDP', 'HTTP']
df.loc[~df['protocol'].isin(valid_protocols), 'protocol'] = df['protocol'].mode()[0]

# ── 2. Source IP Fallback ─────────────────────────────────────────────────────
# IPs cannot be meaningfully imputed (no statistical relationship applies).
# Use 0.0.0.0 as a sentinel/placeholder value — a reserved, non-routable address
# that clearly signals "unknown origin" without breaking IP format expectations.
df['source_ip'] = df['source_ip'].fillna('0.0.0.0')

# ── 3. Port Range Enforcement ─────────────────────────────────────────────────
# clip() enforces hard boundaries on destination_port:
# - Values below 0 → set to 0
# - Values above 65535 → set to 65535 (e.g. the 999999 outlier we saw earlier)
# This is safer than dropping, as the row may still contain valid other fields.
df['destination_port'] = df['destination_port'].clip(lower=0, upper=65535)

# ── Verify ────────────────────────────────────────────────────────────────────
display(df.isnull().sum())
display(df.describe(include='all'))

### Conclusion
Here's what changed vs before:

`source_ip`

- count went from 85 → 100 ✅
- top is now 0.0.0.0 with freq=15 — the 15 invalid IPs got filled with the sentinel value as expected

`destination_port`

- max dropped from 999999 → 65535 ✅ — the clip() worked, that outlier got capped at the valid port ceiling
- mean dropped dramatically from 10121 → 776 — because 999999 was massively skewing the average, now gone
- std dropped from 99987 → 6542 — much healthier, data is far less spread out now

`protocol`

- unique dropped from 10 → 9 — one invalid protocol was replaced by the mode
- freq for HTTP went from 25 → 27 — confirming 2 invalid protocols were replaced with HTTP

Everything else:

- bytes_transferred and threat_level unchanged — they were already clean after KNN imputation
- count is 100 across all columns ✅ — no NaN anywhere

**Bottom line**: the pipeline worked end to end — validation → imputation → cleanup → fully populated, range-enforced, format-valid dataset. 🎯